<a href="https://colab.research.google.com/github/imanuni/imanuni/blob/main/arabBert-test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install transformers datasets torch -q

In [2]:
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [12]:

from google.colab import files

# Sélectionne ton fichier local
uploaded = files.upload()
import os
print(os.listdir())  # Vérifie que model.xlsx est bien là
df = pd.read_csv("model-clean.csv", encoding="utf-8")
print("📌 Données brutes :")
print(df.head())

Saving model-clean.csv to model-clean (2).csv
['.config', 'model-clean.csv', 'model-clean (2).csv', 'model-clean (1).csv', 'sample_data']
📌 Données brutes :
                                               titre label
0     عون: لحكومة تمثل الجميع مطالب كتل تؤخر التأليف   yes
1  زيارة إيرانية للراعي: رسالة وتطمينات وسجادة بق...    no
2                 تفاؤل بحل قضية الودائع: على أي أسس    no
3  بوادر حملة عسكرية إسرائيلية غربية: صنعاء تتحضر...    no
4  وزير الخارجية السعودي إلى بيروت بين الخميسين و...    no


In [15]:
df["label"] = df["label"].map({"yes": 1, "no": 0})

print("\n📌 Après conversion yes/no → 0/1 :")
print(df.head())


📌 Après conversion yes/no → 0/1 :
                                               titre  label
0     عون: لحكومة تمثل الجميع مطالب كتل تؤخر التأليف      1
1  زيارة إيرانية للراعي: رسالة وتطمينات وسجادة بق...      0
2                 تفاؤل بحل قضية الودائع: على أي أسس      0
3  بوادر حملة عسكرية إسرائيلية غربية: صنعاء تتحضر...      0
4  وزير الخارجية السعودي إلى بيروت بين الخميسين و...      0


In [18]:
 #Création Dataset HuggingFace

dataset = Dataset.from_pandas(df)

print("\n Exemple dataset brut (5 lignes) :")
print("\n",dataset[:5])


 Exemple dataset brut (5 lignes) :

 {'titre': ['عون: لحكومة تمثل الجميع مطالب كتل تؤخر التأليف', 'زيارة إيرانية للراعي: رسالة وتطمينات وسجادة بقطب مخفية', 'تفاؤل بحل قضية الودائع: على أي أسس', 'بوادر حملة عسكرية إسرائيلية غربية: صنعاء تتحضر لتصعيد وشيك', 'وزير الخارجية السعودي إلى بيروت بين الخميسين وتشاؤم مسيحي حول صعود الدخان الأبيض الرئاسي'], 'label': [1, 0, 0, 0, 0]}


In [19]:
 #Tokenisation avec AraBERT

model_name = "aubmindlab/bert-base-arabertv02"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):
    return tokenizer(
        example["titre"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

dataset = dataset.map(tokenize_function, batched=True)

print("\n📌 Exemple dataset après tokenisation (5 lignes) :")
print(dataset[:5])

Map:   0%|          | 0/1113 [00:00<?, ? examples/s]


📌 Exemple dataset après tokenisation (5 lignes) :
{'titre': ['عون: لحكومة تمثل الجميع مطالب كتل تؤخر التأليف', 'زيارة إيرانية للراعي: رسالة وتطمينات وسجادة بقطب مخفية', 'تفاؤل بحل قضية الودائع: على أي أسس', 'بوادر حملة عسكرية إسرائيلية غربية: صنعاء تتحضر لتصعيد وشيك', 'وزير الخارجية السعودي إلى بيروت بين الخميسين وتشاؤم مسيحي حول صعود الدخان الأبيض الرئاسي'], 'label': [1, 0, 0, 0, 0], 'input_ids': [[2, 6393, 31, 14494, 3228, 1995, 3772, 30663, 49599, 28698, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 2764, 22296, 27705, 578, 31, 3249, 6520, 33828, 667, 23698, 197, 1211, 1381, 1350, 1719, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 29188, 10694, 2205, 15542, 31, 323, 559, 6910, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [23]:
# 7. Conversion de label en ClassLabel
from datasets import Dataset, ClassLabel
class_labels = ClassLabel(num_classes=2, names=["autre", "declaration"])
dataset = dataset.cast_column("label", class_labels)

# Split train/test avec stratification
dataset = dataset.train_test_split(test_size=0.2, stratify_by_column="label")

print("\n📌 Train set (5 lignes) :")
print(dataset["train"][:5])


Casting the dataset:   0%|          | 0/1113 [00:00<?, ? examples/s]


📌 Train set (5 lignes) :
{'titre': ['بري: قد نصبح امام مشكلتي الرئاسة والانتخابات النيابية', 'ـ واشنطن: تأجيل الانسحاب موقتا', 'سلام ممتعض ولن يعتذر', 'ـ السيد نصرالله في عيد التحرير: كلما ضعفت إسرائيل ساد الأمن والأمان بلبنان', 'المقاومة تراقب منطق الضمانات الدولية: المتغير السوري عزز عدوانية العدو'], 'label': [1, 1, 0, 1, 0], 'input_ids': [[2, 3225, 31, 602, 51269, 2026, 57760, 3621, 28987, 6546, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 1, 2646, 31, 5682, 7492, 58974, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 2766, 50547, 235, 3387, 27583, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 1, 2002, 160

In [25]:
#Tokenisation avec AraBERT
model_name = "aubmindlab/bert-base-arabertv02"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):
    return tokenizer(
        example["titre"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

dataset = Dataset.from_pandas(df)
dataset = dataset.map(tokenize_function, batched=True)
dataset = dataset.map(tokenize_function, batched=True)
print("📌 Exemple après tokenisation :")
print(dataset[:5])



Map:   0%|          | 0/1113 [00:00<?, ? examples/s]

Map:   0%|          | 0/1113 [00:00<?, ? examples/s]

📌 Exemple après tokenisation :
{'titre': ['عون: لحكومة تمثل الجميع مطالب كتل تؤخر التأليف', 'زيارة إيرانية للراعي: رسالة وتطمينات وسجادة بقطب مخفية', 'تفاؤل بحل قضية الودائع: على أي أسس', 'بوادر حملة عسكرية إسرائيلية غربية: صنعاء تتحضر لتصعيد وشيك', 'وزير الخارجية السعودي إلى بيروت بين الخميسين وتشاؤم مسيحي حول صعود الدخان الأبيض الرئاسي'], 'label': [1, 0, 0, 0, 0], 'input_ids': [[2, 6393, 31, 14494, 3228, 1995, 3772, 30663, 49599, 28698, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 2764, 22296, 27705, 578, 31, 3249, 6520, 33828, 667, 23698, 197, 1211, 1381, 1350, 1719, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 29188, 10694, 2205, 15542, 31, 323, 559, 6910, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [9]:
# Conversion yes -> 1, no -> 0
df["label"] = df["label"].map({"yes": 1, "no": 0})

print(df.head())

                                               titre  label
0     عون: لحكومة تمثل الجميع مطالب كتل تؤخر التأليف      1
1  زيارة إيرانية للراعي: رسالة وتطمينات وسجادة بق...      0
2                 تفاؤل بحل قضية الودائع: على أي أسس      0
3  بوادر حملة عسكرية إسرائيلية غربية: صنعاء تتحضر...      0
4  وزير الخارجية السعودي إلى بيروت بين الخميسين و...      0


In [26]:
# 8. Charger le modèle
# ================================
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [28]:
# 9. Définir métriques

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        p.label_ids, preds, average="binary"
    )
    acc = accuracy_score(p.label_ids, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


In [29]:
# 10. Configurer l’entraînement
# ================================
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
)

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'